Connect to Google Drive and Create MICOM Folder

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Creat main folder for project inputs and outputs
import os
micom_path = '/content/drive/MyDrive/MICOM'
os.makedirs(micom_path, exist_ok=True)

Install MICOM and AGORA Database

In [ ]:
# Install MICOM and dependencies
!pip install micom
!pip install cobra optlang swiglpk

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import shutil

import cobra
from micom.workflows import build, build_database
from micom import load_pickle

In [ ]:
# Install AGORA
# Use the shell command !wget to download AGORA2 models as a zip file
!wget "https://www.vmh.life/files/reconstructions/AGORA2/version2.01/sbml_files_fixed/zipped/AGORA2_models/AGORA2_SBML.zip"

--2025-11-30 19:13:29--  https://www.vmh.life/files/reconstructions/AGORA2/version2.01/sbml_files_fixed/zipped/AGORA2_models/AGORA2_SBML.zip
Resolving www.vmh.life (www.vmh.life)... 145.239.0.239
Connecting to www.vmh.life (www.vmh.life)|145.239.0.239|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2170499647 (2.0G) [application/zip]
Saving to: ‘AGORA2_SBML.zip’

AGORA2_SBML.zip     100%[===================>]   2.02G  36.9MB/s    in 57s     

2025-11-30 19:14:27 (36.2 MB/s) - ‘AGORA2_SBML.zip’ saved [2170499647/2170499647]



In [ ]:
# Define the path for the AGORA Database
agora_path = Path(micom_path) / "AGORA2_models"

In [ ]:
# Extract the models
!unzip AGORA2_SBML.zip -d {agora_path}

Streaming output truncated to the last 5000 lines.
  inflating: /content/drive/MyDrive/MICOM/AGORA2_models/Enterococcus_faecium_9830512_2.xml  
  inflating: /content/drive/MyDrive/MICOM/AGORA2_models/Enterococcus_faecium_9830565_4.xml  
  inflating: /content/drive/MyDrive/MICOM/AGORA2_models/Enterococcus_faecium_9930238_2.xml  
  inflating: /content/drive/MyDrive/MICOM/AGORA2_models/Enterococcus_faecium_9931110_4.xml  
  inflating: /content/drive/MyDrive/MICOM/AGORA2_models/Enterococcus_faecium_A17_Sv1.xml  
  inflating: /content/drive/MyDrive/MICOM/AGORA2_models/Enterococcus_faecium_ATCC_8459.xml  
  inflating: /content/drive/MyDrive/MICOM/AGORA2_models/Enterococcus_faecium_Aus0004.xml  
  inflating: /content/drive/MyDrive/MICOM/AGORA2_models/Enterococcus_faecium_C309.xml  
  inflating: /content/drive/MyDrive/MICOM/AGORA2_models/Enterococcus_faecium_C68.xml  
  inflating: /content/drive/MyDrive/MICOM/AGORA2_models/Enterococcus_faecium_Com12.xml  
  inflating: /content/drive/MyDrive/MI

In [ ]:
# Remove zip file
!rm AGORA2_SBML.zip

Build Class for MICOM Pre-processing and Analysis


In [ ]:
# Create a class to handle pre-processing and analysis functions.
class MICOM_Analysis:
#-------------------------------------------------------------------------------------------------------------------------------------------
  def __init__(self, rel_abd, condition):
    '''
    Constructor Method: Creates instance of the class with the following attributes:
          self - reference to the class object
          rel_abd - dataframe of relative abundance data
          condition - string of condition name
    '''
    self.rel_abd = rel_abd
    self.condition = condition

    # Creat a subfolder in MICOM folder named /condition_communities
    # Will be used to store .pickle files
    self.community_folder = Path(micom_path) / f"{self.condition}_communities"
    os.makedirs(self.community_folder, exist_ok=True)

    # Create an attribute to store the community manifest
    self.community_manifest = None
#-------------------------------------------------------------------------------------------------------------------------------------------
  def format_rel_abd(self):
    '''
    Pre-Processing Method: Formats the relative abundance dataframe
          self - reference to the class object
          Tasks:
          - Renames the "#OTU ID" column to "id" as required for MICOM analysis
          - Extracts genus names from the "id" column and creates a new "genus" column
          - Reshapes the dataframe to a long format (required by MICOM)
          - Changes "id" column to genus name (required by MICOM)
    '''
    # Rename the "#OTU ID" column to "id" as required for MICOM analysis
    self.rel_abd = self.rel_abd.rename(columns={"#OTU ID": "id"})

    # Create an empty list to store genus names
    genus_col = []
    for microbe in self.rel_abd["id"]:
      # For each microbe in the id column:
      # Split microbe into a list of ranks [domain, phylum, class, order, family, genus, species]
      lineage = microbe.split(';')
      # Initially set genus to "Unknown"
      genus = "Unknown"

      for rank in lineage:
        # For each rank in the lineage, check if it starts with "g__" for genus
        if rank.startswith("g__"):
          # If True, remove "g__" to leave only the genus name
          genus = rank.replace("g__", "")
          # Remove any brackets
          genus = genus.replace('[', '').replace(']', '')
          # If genus name is segmented by "-" or "_", rename to only the first part of the segment
          # Example: "Escherichia-Shigella" → "Escherichia"
          if '-' in genus:
            genus = genus.split('-')[0]
          if '_' in genus:
            genus = genus.split('_')[0]

      # Append genus name to genus_col, if genus was not classified it will remain as "Unknown"
      genus_col.append(genus)

    # Insert a column into the relative abundance dataframe, at index = 1 (second column), named
    # "genus", populated with values from genus_col
    self.rel_abd.insert(loc = 1, column = "genus", value = genus_col)

    # Reshape the dataframe to a long format (required by MICOM)
    # It keeps the "id" and "genus" columns as is
    # It creates a variable column that contains the names of the remaining columns (sample SRR IDs)
    # It creates a value column for the corresponding relative abundance values for each sample
    self.rel_abd = self.rel_abd.melt(
        id_vars = ["id", "genus"],   # remain unchanged
        var_name = "sample_id",      # name of variable column
        value_name = "abundance")    # name of value column

    # Change "id" column to genus name
    self.rel_abd["id"] = self.rel_abd["genus"]
#-------------------------------------------------------------------------------------------------------------------------------------------
  def genus_mapping(self, manifest):
    '''
    Maps each microbe in dataframe to a genus present in the AGORA database
          self - reference to the class object
          genus_database - list of genus names in the AGORA database
          Tasks:
          - Remove rows where the genus is "Unknown"
          - Remove rows where the genus cannot be mapped.

    # Remove sample where genus in Unknown
    self.rel_abd = self.rel_abd[self.rel_abd["genus"] != "Unknown"]
    # Check if sample genus exists in AGORA database, remove samples if not
    self.rel_abd = self.rel_abd[self.rel_abd["genus"].isin(genus_database)]
    # Reset dataframe index
    self.rel_abd = self.rel_abd.reset_index(drop=True)
    '''
    self.rel_abd = self.rel_abd[self.rel_abd["genus"] != "Unknown"].copy()

    # 2. Get allowed genera from manifest
    valid_genera = set(manifest["id"].unique())

    # 3. Keep only taxa that have a GEM
    self.rel_abd = self.rel_abd[self.rel_abd["genus"].isin(valid_genera)].copy()


    # 5. Reset index
    self.rel_abd = self.rel_abd.reset_index(drop=True)
#-------------------------------------------------------------------------------------------------------------------------------------------
  def build_communities(self, gem_folder):
    database_manifest_path = Path(gem_folder) / "manifest.csv"
    database_manifest = pd.read_csv(database_manifest_path)
    database_manifest = database_manifest[["genus", "file"]]

    taxonomy = self.rel_abd.copy()
    taxonomy = taxonomy.merge(
        database_manifest,
        on = "genus",
        how = "left")
    taxonomy = taxonomy[~taxonomy["file"].isna()].reset_index(drop=True)
    self.rel_abd = taxonomy

    self.community_manifest = build(
        taxonomy = self.rel_abd,
        model_db = None,
        out_folder = str(self.community_folder),
        cutoff = 0.0001,
        threads = 2)
#-------------------------------------------------------------------------------------------------------------------------------------------
  def check_communities(self):
    com_manifest_path = Path(self.community_folder) / "manifest.csv"

    if com_manifest_path.exists():
      self.community_manifest = pd.read_csv(com_manifest_path)

    else:
      self.community_manifest = []
      pickles = list(Path(self.community_folder).glob("*.pickle"))
      for community in pickles:
        sample_srr = community.stem
        self.community_manifest.append({
            "sample_id": sample_srr,
            "pickle_file": str(community)})

    self.community_manifest = pd.DataFrame(self.community_manifest)
    self.community_manifest.to_csv(com_manifest_path, index=False)

    self.rel_abd["sample_id"] = self.rel_abd["sample_id"].astype(str)
    valid_samples = set(self.community_manifest["sample_id"].astype(str))
    self.rel_abd = (self.rel_abd[self.rel_abd["sample_id"].isin(valid_samples)].reset_index(drop=True))
#-------------------------------------------------------------------------------------------------------------------------------------------


Format Relative Abundance TSVs

In [ ]:
# Read the relative abundance files into varaibles named after corresponding sample conditions
ad_rel_abd = pd.read_csv('/content/drive/MyDrive/Taxonomic_Identification/ad_species_relative_abundance.tsv', sep="\t")
mci_rel_abd = pd.read_csv('/content/drive/MyDrive/Taxonomic_Identification/mci_species_relative_abundance.tsv', sep="\t")
healthy_rel_abd = pd.read_csv('/content/drive/MyDrive/Taxonomic_Identification/healthy_species_relative_abundance.tsv', sep="\t")

In [ ]:
# Initialize MICOM_Analysis instances for AD, MCI, and Healthy datasets
ad = MICOM_Analysis(ad_rel_abd, "ad")
mci = MICOM_Analysis(mci_rel_abd, "mci")
healthy = MICOM_Analysis(healthy_rel_abd, "healthy")

In [ ]:
# Print the relative abundance datasets before pre-processing
print("ad", ad.rel_abd)
print("mci", mci.rel_abd)
print("healthy", healthy.rel_abd)

ad                                                #OTU ID  SRR8061715  \
0    d__Bacteria;p__Proteobacteria;c__Gammaproteoba...    0.000000   
1    d__Bacteria;p__Firmicutes;c__Clostridia;o__Lac...    0.000000   
2    d__Bacteria;p__Actinobacteriota;c__Actinobacte...    0.105002   
3    d__Bacteria;p__Firmicutes;c__Clostridia;o__Lac...    0.033799   
4    d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...    0.005408   
..                                                 ...         ...   
101  d__Bacteria;p__Firmicutes;c__Clostridia;o__Osc...    0.000000   
102  d__Bacteria;p__Proteobacteria;c__Gammaproteoba...    0.000000   
103  d__Bacteria;p__Fusobacteriota;c__Fusobacteriia...    0.000000   
104  d__Bacteria;p__Firmicutes;c__Clostridia;o__Osc...    0.000000   
105  d__Bacteria;p__Firmicutes;c__Clostridia;o__Osc...    0.000000   

     SRR8061716  SRR8061717  SRR8061718  SRR8061719  SRR8061720  SRR8061721  \
0      0.509391    0.294271    0.841240    0.000000    0.147904    0.648108  

In [ ]:
# Format relative abundance tables to correct MICOM format, remove rows where genus is unknown
ad.format_rel_abd()
mci.format_rel_abd()
healthy.format_rel_abd()

In [ ]:
# Print the relative abundance datasets after pre-processing
print("ad", ad.rel_abd)
print("mci", mci.rel_abd)
print("healthy", healthy.rel_abd)

ad                      id              genus   sample_id  abundance
0           Escherichia        Escherichia  SRR8061715   0.000000
1          Ruminococcus       Ruminococcus  SRR8061715   0.000000
2       Bifidobacterium    Bifidobacterium  SRR8061715   0.105002
3           Eubacterium        Eubacterium  SRR8061715   0.033799
4     Ligilactobacillus  Ligilactobacillus  SRR8061715   0.005408
...                 ...                ...         ...        ...
3493   Colidextribacter   Colidextribacter  SRR8061777   0.000000
3494          Neisseria          Neisseria  SRR8061777   0.000000
3495      Fusobacterium      Fusobacterium  SRR8061777   0.000000
3496             DTU089             DTU089  SRR8061777   0.000000
3497         Paludicola         Paludicola  SRR8061777   0.000000

[3498 rows x 4 columns]
mci                    id            genus   sample_id  abundance
0         Eubacterium      Eubacterium  SRR8061735   0.000000
1         Bacteroides      Bacteroides  SRR8061735  

In [ ]:
# Save files for relative abundance datasets after pre-processing and before
# GEM filtration
ad.rel_abd.to_csv(Path(micom_path)/ "ad_rel_abd_frmt.csv", index=False)
mci.rel_abd.to_csv(Path(micom_path)/ "mci_rel_abd_frmt.csv", index=False)
healthy.rel_abd.to_csv(Path(micom_path)/ "healthy_rel_abd_frmt.csv", index=False)

Build Genera Database

In [ ]:
# List all the files in the AGORA database
agora_files = list(agora_path.glob("*.xml"))
# Initialize an empty list for the genera database
genera_database = []
# Create a folder for the GEM database
gem_folder = Path(micom_path) / "GEM_Database"
os.makedirs(gem_folder, exist_ok=True)

In [ ]:
def build_genera_database(ad, mci, healthy):
  '''
  Add all unique genera from AD, MCI, and Healthy datasets to the genera database
       ad - dataframe of relative abundance data from AD dataset
  	   mci - dataframe of relative abundance data from MCI dataset
  	   healthy - dataframe of relative abundance data from Healthy dataset
  '''
  for genus in ad:
    if pd.notna(genus) and genus != "Unknown" and genus not in genera_database:
      genera_database.append(genus)

  for genus in mci:
    if pd.notna(genus) and genus != "Unknown" and genus not in genera_database:
      genera_database.append(genus)

  for genus in healthy:
    if pd.notna(genus) and genus != "Unknown" and genus not in genera_database:
      genera_database.append(genus)

In [ ]:
# Call the build_genera_database using the "genus" columns from each dataframe
build_genera_database(ad.rel_abd["genus"], mci.rel_abd["genus"], healthy.rel_abd["genus"])

In [ ]:
# Print the genera database
print(genera_database)

['Escherichia', 'Ruminococcus', 'Bifidobacterium', 'Eubacterium', 'Ligilactobacillus', 'Blautia', 'Fusicatenibacter', 'Limosilactobacillus', 'Bacteroides', 'Desulfovibrio', 'Phascolarctobacterium', 'Veillonella', 'Anaerostipes', 'Romboutsia', 'Monoglobus', 'Agathobacter', 'Faecalibacterium', 'Megamonas', 'Streptococcus', 'Subdoligranulum', 'UCG', 'Dorea', 'Roseburia', 'Butyricicoccus', 'Lachnospira', 'Prevotella', 'Flavonifractor', 'Lactobacillus', 'Alistipes', 'Coprococcus', 'Enterobacter', 'Tyzzerella', 'Parabacteroides', 'Enterococcus', 'Parasutterella', 'Clostridium', 'Lachnospiraceae', 'Clostridia', 'UBA1819', 'Oscillibacter', 'Raoultibacter', 'Acidaminococcus', 'Turicibacter', 'Intestinibacter', 'Odoribacter', 'Hungatella', 'Dialister', 'Megasphaera', 'Eisenbergiella', 'Pyramidobacter', 'Cellulosilyticum', 'NK4A214', 'Porphyromonas', 'Christensenellaceae', 'Incertae', 'Lachnoclostridium', 'Haemophilus', 'Comamonas', 'Adlercreutzia', 'Negativibacillus', 'uncultured', 'Lactococcus'

Build GEM Database

In [ ]:
def build_gem_database(agora, database):
  '''
  Build a GEM database (manifest) consisting only of files found in the database
       agora - list of files
       database - list of genera in the database
  '''
  manifest = []

  for gem_model in agora:
    # For each GEM model file extract the microbe
    microbe = gem_model.stem
    # Split the microbe into ranks
    lineage = microbe.split("_")

    # Set genus to first rank in lineage
    genus = lineage[0]
    # Set species to second rank in lineage
    species = lineage[1]

    if genus in database:
      # If the genus is in the database, create a dictionary manifest with the above
      # information and append it to the manifest list
      manifest.append({
          "file": str(gem_model),
          "kingdom": "Bacteria",
          "phylum": "",
          "class": "",
          "order": "",
          "family": "",
          "genus": genus,
          "species": species})

  # Change manifest list to dataframe and return
  manifest = pd.DataFrame(manifest)
  return manifest

In [ ]:
# Build a manifest of AGORA GEM models that match the genera present in our samples
database_manifest = build_gem_database(agora_files, genera_database)

In [ ]:
# Use MICOM's build_database() method to generate .json files for each GEM in the filtered AGORA manifest
build_database(
    database_manifest,
    out_path = str(gem_folder),
    rank = "genus",
    threads = 2)

Running ━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   4% -:--:--

KeyboardInterrupt: 

Process Missing GEMS

In [ ]:
def check_missing_gems():
  '''
  Check which required genera (present in sample datasets) are missing from the GEM database
  '''
  # List all files in the GEM database
  gem_files = list(gem_folder.glob("*.json"))
  # Create empty lists to store genera present and missing in the GEM database
  gem_database = []
  missing_gems = []

  # For each unique GEM in the GEM database, add it to gem_database list
  for gem in gem_files:
    gem = gem.stem
    if gem not in gem_database:
      gem_database.append(gem)

  # Check if each genus in the genera database (from sample datasets) has a corresponding
  # GEM model in the GEM database
  for genus in genera_database:
    # If not add to list of missing_gems
    if genus not in gem_database:
      missing_gems.append(genus)

    # If genus is present in both the GEM database and missing_gems list (if it was added later)
    # remove it from the missing_gems list
    elif genus in gem_database:
      if genus in missing_gems:
        missing_gems.remove(genus)

  # If there are no missing GEMS print a message that "All GEMs Found"
  if len(missing_gems) == 0:
    print("All GEMs Found")
  # Else print each of the genera missing GEM models
  else:
    for missing in missing_gems:
      print("GEM Not Found: ", missing)

  return missing_gems

In [ ]:
# Check for missing GEM models
missing_gems = check_missing_gems()

GEM Not Found:  Ligilactobacillus
GEM Not Found:  Limosilactobacillus
GEM Not Found:  Monoglobus
GEM Not Found:  Agathobacter
GEM Not Found:  UCG
GEM Not Found:  Clostridia
GEM Not Found:  UBA1819
GEM Not Found:  Raoultibacter
GEM Not Found:  Hungatella
GEM Not Found:  Megasphaera
GEM Not Found:  NK4A214
GEM Not Found:  Incertae
GEM Not Found:  Negativibacillus
GEM Not Found:  Conservatibacter
GEM Not Found:  Peptococcus
GEM Not Found:  Holdemanella
GEM Not Found:  Muribaculaceae
GEM Not Found:  Chloroplast
GEM Not Found:  Family
GEM Not Found:  Vicinamibacteraceae
GEM Not Found:  Colidextribacter
GEM Not Found:  DTU089
GEM Not Found:  Paludicola
GEM Not Found:  Tannerellaceae
GEM Not Found:  CAG
GEM Not Found:  Tuzzerella
GEM Not Found:  Epulopiscium
GEM Not Found:  P5D1
GEM Not Found:  Izemoplasmatales
GEM Not Found:  Hydrogenoanaerobacterium
GEM Not Found:  TM7x
GEM Not Found:  RF39


In [ ]:
def process_missing_gems(missing_database):
  '''
  Process AGORA database for missing and broken GEM files
        missing_database - list of genera missing from the GEM database
        Tasks:
        - Find whether missing GEM models have available or missing xml files
        - Check if available xml files are broken
        - Copy valid available xml files into a separate folder
        - Remove broken available xml files
        - Build GEM database manifest with only valid available xml files
  '''
  # Create empty list to store Path objects for available xml files
  available_xml = []
  # Create empty list to store genera with missing xml files
  missing_xml = []

  # For each missing GEM
  for gem in missing_database:
    found = False
    # Compare missing GEM to all xml files in AGORA database
    for xml in agora_files:
      xml_genus = xml.stem.split("_")[0]
      # If there are available files for the GEM add to available_xml list
      if gem == xml_genus:
        available_xml.append(xml)
        found = True
        break
    # If there are no matching xml files for the missing GEM add to missing_xml list
    if not found:
      missing_xml.append(gem)
#-------------------------------------------------------------------------------------------------------------------------------------------
  # Create empty lists to store Path objects for valid and broken available xml files
  valid_xml = []
  broken_xml = []

  # For each available xml, try to load it using COBRA
  for xml in available_xml:
    try:
      cobra.io.read_sbml_model(str(xml))
      # If it loads successfully, add it to list of valid_xml
      valid_xml.append(xml)
    # If an error is raised, print the error and Path of broken file
    # Add xml to list of broken_xml
    except Exception as error:
      broken_xml.append(xml)
      print(error, "\n")
      print("Broken SBML: ", xml, "\n")
#-------------------------------------------------------------------------------------------------------------------------------------------
  # Create folders to store valid and broken xml files
  valid_folder = Path(micom_path) / "Valid Missing xmls"
  broken_folder = Path(micom_path) / "Broken xmls"
  valid_folder.mkdir(exist_ok=True)
  broken_folder.mkdir(exist_ok=True)

  # Copy valid xmls into a separate folder
  for xml_file in valid_xml:
    valid_destination = Path(valid_folder) / xml_file.name
    shutil.copy(xml_file, valid_destination)

  # Move broken xml files into a separate folder
  for xml_file in broken_xml:
    broken_destination = Path(broken_folder) / xml_file.name
    xml_file.rename(broken_destination)

  # Build a GEM manifest from the valid xml GEM models and return it along with any genera still missing xml files
  manifest = build_gem_database(valid_xml, missing_database)
  return [manifest, missing_xml]

In [ ]:
missing_manifest = process_missing_gems(missing_gems)

In [ ]:
# Print list of GEM models missing xml files
print(missing_manifest[1])

['Ligilactobacillus', 'Limosilactobacillus', 'Monoglobus', 'Agathobacter', 'UCG', 'Clostridia', 'UBA1819', 'Raoultibacter', 'Hungatella', 'NK4A214', 'Incertae', 'Negativibacillus', 'Conservatibacter', 'Peptococcus', 'Holdemanella', 'Muribaculaceae', 'Chloroplast', 'Family', 'Vicinamibacteraceae', 'Colidextribacter', 'DTU089', 'Paludicola', 'Tannerellaceae', 'CAG', 'Tuzzerella', 'Epulopiscium', 'P5D1', 'Izemoplasmatales', 'Hydrogenoanaerobacterium', 'TM7x', 'RF39']


In [ ]:
# Use MICOM's build_database() method to generate .json files for the missing GEMs with valid xml files
build_database(
    missing_manifest[0],
    out_path = str(gem_folder),
    rank = "genus",
    threads = 2)

Output()

,file,kingdom,phylum,class,order,family,genus,species,id,summary_rank
genus,,,,,,,,,,
Acinetobacter,Acinetobacter.json,Bacteria,,,,,Acinetobacter,baumannii,Acinetobacter,genus
Alistipes,Alistipes.json,Bacteria,,,,,Alistipes,finegoldii,Alistipes,genus
Anaerotruncus,Anaerotruncus.json,Bacteria,,,,,Anaerotruncus,colihominis,Anaerotruncus,genus
Bacteroides,Bacteroides.json,Bacteria,,,,,Bacteroides,acidifaciens,Bacteroides,genus
Bifidobacterium,Bifidobacterium.json,Bacteria,,,,,Bifidobacterium,adolescentis,Bifidobacterium,genus
Citrobacter,Citrobacter.json,Bacteria,,,,,Citrobacter,amalonaticus,Citrobacter,genus
Clostridium,Clostridium.json,Bacteria,,,,,Clostridium,acetobutylicum,Clostridium,genus
Collinsella,Collinsella.json,Bacteria,,,,,Collinsella,aerofaciens,Collinsella,genus
Desulfovibrio,Desulfovibrio.json,Bacteria,,,,,Desulfovibrio,desulfuricans,Desulfovibrio,genus


In [ ]:
# Check for missing GEM models
missing_gems = check_missing_gems()

GEM Not Found:  Ligilactobacillus
GEM Not Found:  Limosilactobacillus
GEM Not Found:  Monoglobus
GEM Not Found:  Agathobacter
GEM Not Found:  UCG
GEM Not Found:  Clostridia
GEM Not Found:  UBA1819
GEM Not Found:  Raoultibacter
GEM Not Found:  Hungatella
GEM Not Found:  Megasphaera
GEM Not Found:  NK4A214
GEM Not Found:  Incertae
GEM Not Found:  Negativibacillus
GEM Not Found:  Conservatibacter
GEM Not Found:  Peptococcus
GEM Not Found:  Holdemanella
GEM Not Found:  Muribaculaceae
GEM Not Found:  Chloroplast
GEM Not Found:  Family
GEM Not Found:  Vicinamibacteraceae
GEM Not Found:  Colidextribacter
GEM Not Found:  DTU089
GEM Not Found:  Paludicola
GEM Not Found:  Tannerellaceae
GEM Not Found:  CAG
GEM Not Found:  Tuzzerella
GEM Not Found:  Epulopiscium
GEM Not Found:  P5D1
GEM Not Found:  Izemoplasmatales
GEM Not Found:  Hydrogenoanaerobacterium
GEM Not Found:  TM7x
GEM Not Found:  RF39


Update Database Manifest


In [ ]:
# List all GEMs in the final database
final_gems = list(gem_folder.glob("*.json"))

# Initialize an empty manifest
manifest = []
# For each GEM in the database, store its information in a dictionary and append to manifest
for gem in final_gems:
  genus = gem.stem
  manifest.append({
      "id": genus,
      "file": str(gem),
      "kingdom": "Bacteria",
      "phylum": "",
      "class": "",
      "order": "",
      "family": "",
      "genus": genus,
      "species": "",
      "summary_rank": "genus",
      "type": "json",
    })

# Creat database_manifest by converting manifest list into dataframe
database_manifest = pd.DataFrame(manifest)
# Ensure all file Paths are stored as strings
database_manifest["file"] = database_manifest["file"].astype(str)
# Save database_manifest as both a .json and .csv file in the GEM database folder
database_manifest.to_csv(gem_folder / "manifest.csv", index = False)

In [ ]:
# Map genera from dataframes to AGORA database, remove unmapped genera
ad.genus_mapping(database_manifest)
mci.genus_mapping(database_manifest)
healthy.genus_mapping(database_manifest)

In [ ]:
# Print final relative abundance tables
print("ad", ad.rel_abd)
print("mci", mci.rel_abd)
print("healthy", healthy.rel_abd)

ad                    id            genus   sample_id  abundance
0         Escherichia      Escherichia  SRR8061715   0.000000
1        Ruminococcus     Ruminococcus  SRR8061715   0.000000
2     Bifidobacterium  Bifidobacterium  SRR8061715   0.105002
3         Eubacterium      Eubacterium  SRR8061715   0.033799
4             Blautia          Blautia  SRR8061715   0.061740
...               ...              ...         ...        ...
2404      Eubacterium      Eubacterium  SRR8061777   0.000000
2405      Eubacterium      Eubacterium  SRR8061777   0.000000
2406      Eggerthella      Eggerthella  SRR8061777   0.000000
2407        Neisseria        Neisseria  SRR8061777   0.000000
2408    Fusobacterium    Fusobacterium  SRR8061777   0.000000

[2409 rows x 4 columns]
mci                          id                  genus   sample_id  abundance
0               Eubacterium            Eubacterium  SRR8061735   0.000000
1               Bacteroides            Bacteroides  SRR8061735   0.095480
2 

Build Communities

In [ ]:
# Build MICOM community models for the AD samples
ad.build_communities(gem_folder)

[12/05/25 15:30:30] WARNING  Found existing models for 11 samples. Will skip those. Delete the output   ]8;id=49166;file:///usr/local/lib/python3.12/dist-packages/micom/workflows/build.py\build.py]8;;\:]8;id=780356;file:///usr/local/lib/python3.12/dist-packages/micom/workflows/build.py#98\98]8;;\
                             folder if you would like me to rebuild them.                                          

Output()

KeyboardInterrupt: 

In [ ]:
# Build community manifest
ad.check_communities()

In [ ]:
print(ad.rel_abd)

                  id            genus   sample_id  abundance
0        Escherichia      Escherichia  SRR8061715   0.000000
1       Ruminococcus     Ruminococcus  SRR8061715   0.000000
2    Bifidobacterium  Bifidobacterium  SRR8061715   0.105002
3        Eubacterium      Eubacterium  SRR8061715   0.033799
4            Blautia          Blautia  SRR8061715   0.061740
..               ...              ...         ...        ...
798      Eubacterium      Eubacterium  SRR8061777   0.000000
799      Eubacterium      Eubacterium  SRR8061777   0.000000
800      Eggerthella      Eggerthella  SRR8061777   0.000000
801        Neisseria        Neisseria  SRR8061777   0.000000
802    Fusobacterium    Fusobacterium  SRR8061777   0.000000

[803 rows x 4 columns]


In [ ]:
# Build MICOM community models for the MCI samples
mci.build_communities(gem_folder)

In [ ]:
# Build community manifest
mci.check_communities()

In [ ]:
print(mci.rel_abd)

                        id                  genus   sample_id  abundance
0              Eubacterium            Eubacterium  SRR8061735   0.000000
1              Bacteroides            Bacteroides  SRR8061735   0.095480
2          Bifidobacterium        Bifidobacterium  SRR8061735   0.006919
3             Anaerostipes           Anaerostipes  SRR8061735   0.000000
4    Phascolarctobacterium  Phascolarctobacterium  SRR8061735   0.015683
..                     ...                    ...         ...        ...
611            Citrobacter            Citrobacter  SRR8061802   0.000000
612              Bilophila              Bilophila  SRR8061802   0.000000
613              Comamonas              Comamonas  SRR8061802   0.000000
614          Adlercreutzia          Adlercreutzia  SRR8061802   0.000000
615        Christensenella        Christensenella  SRR8061802   0.000000

[616 rows x 4 columns]


In [ ]:
# Build MICOM community models for the Healthy samples
healthy.build_communities(gem_folder)

In [ ]:
# Build community manifest
healthy.check_communities()

In [ ]:
print(healthy.rel_abd)

                  id            genus   sample_id  abundance
0        Escherichia      Escherichia  SRR8061749   0.792208
1        Eubacterium      Eubacterium  SRR8061749   0.000000
2       Anaerostipes     Anaerostipes  SRR8061749   0.000000
3            Blautia          Blautia  SRR8061749   0.066434
4    Bifidobacterium  Bifidobacterium  SRR8061749   0.000000
..               ...              ...         ...        ...
562      Eubacterium      Eubacterium  SRR8061782   0.000000
563      Oxalobacter      Oxalobacter  SRR8061782   0.000000
564        Bilophila        Bilophila  SRR8061782   0.000000
565  Christensenella  Christensenella  SRR8061782   0.000000
566           Rothia           Rothia  SRR8061782   0.000000

[567 rows x 4 columns]


Identify Key Metabolites

In [ ]:
def run_cooperative_tradeoff(community, fraction = 0.7, fluxes = False, pfba = True):
  cooperative_tradeoff = community.cooperative_tradeoff(
      fraction = fraction,
      fluxes = fluxes,
      pfba = pfba)

  return cooperative_tradeoff

def test_community_flux(community_list):
  com_fluxes = []
  sample_srr = []

  for community in community_list:
    test_community = load_pickle(community)
    # Run FBA analysis to get metabolic fluxes
    test_solution = run_cooperative_tradeoff(test_community, fraction = 0.7, fluxes = True, pfba = True)
    # Save table of flux (rows = taxa/medium, cols = reactions)
    fluxes = test_solution.fluxes

    fluxes = fluxes.loc["medium"].dropna()
    sorted_flux = fluxes.reindex(fluxes.abs().sort_values(ascending=False).index)

    com_fluxes.append(sorted_flux)
    sample_srr.append(community.stem)
    print(len(sample_srr))

  com_fluxes = pd.DataFrame(com_fluxes, index=sample_srr)
  return com_fluxes

def reaction_counts(com_fluxes):
  counts = com_fluxes.notna().sum().sort_values(ascending=False)
  return counts

def flux_ratios(ad_fluxes, mci_fluxes, healthy_fluxes):
  # ratio of metabolite flux being present in a sample
  counts = pd.DataFrame({
      "AD_count": ad_fluxes.notna().sum(),
      "MCI_count": mci_fluxes.notna().sum(),
      "Healthy_count": healthy_fluxes.notna().sum()})

  ratios = pd.DataFrame({
      "AD_ratio": counts["AD_count"] / ad_fluxes.shape[0],
      "MCI_ratio": counts["MCI_count"] / mci_fluxes.shape[0],
      "Healthy_ratio": counts["Healthy_count"] / healthy_fluxes.shape[0]})

  # Find difference btw max and min ratio, if ratio for each groups is similar
  # range will be small, if ratio for each group is different, range will be large
  ratios["presence_range"] = ratios.max(axis = 1) - ratios.min(axis = 1)
  ratios = ratios.sort_values("presence_range", ascending=False)

  return ratios

def community_growth_table(community_list):
  """
  Compute community growth rate using cooperative tradeoff.
  Returns a DataFrame indexed by sample_id with one column:
      - community_growth
  """

  rows = []

  for sample in community_list:
    sample_srr = sample.stem
    print(sample_srr)

    # Load community
    community = load_pickle(sample)

    # Run cooperative tradeoff
    tradeoff_solution = run_cooperative_tradeoff(community, fraction = 0.7, fluxes = False, pfba = True)

    # Extract growth safely (handle different MICOM versions)
    if hasattr(tradeoff_solution, "community_growth"):
      growth = tradeoff_solution.community_growth
    elif hasattr(tradeoff_solution, "growth_rate"):
      growth = tradeoff_solution.growth_rate
    else:
      growth = None

    rows.append({
        "sample_id": sample_srr,
        "community_growth": float(growth) if growth is not None else None})

  growth_df = pd.DataFrame(rows).set_index("sample_id")
  return growth_df

In [ ]:
def run_micom_analysis(condition):
  community_list = list(Path(condition.community_folder).glob("*.pickle"))
  fluxes = test_community_flux(community_list)
  counts = reaction_counts(fluxes)
  growth = community_growth_table(community_list)

  return fluxes, counts, growth

In [ ]:
ad_micom_analysis = run_micom_analysis(ad)
ad_fluxes = ad_micom_analysis[0]
ad_counts = ad_micom_analysis[1]
ad_growth = ad_micom_analysis[2]

1
2
3
4
5
6
7
8
9
10
11
SRR8061716
SRR8061715
SRR8061718
SRR8061722
SRR8061719
SRR8061732
SRR8061734
SRR8061743
SRR8061771
SRR8061772
SRR8061777


In [ ]:
print(ad_fluxes.head())

reaction       EX_ac_m    EX_co2_m  EX_ala_L_m   EX_etoh_m   EX_xtsn_m  \
SRR8061716  690.446042  672.789572  644.813891  325.169243  194.772826   
SRR8061715   -0.146698   99.679088   -0.240389    8.448048         NaN   
SRR8061718  878.207937  667.686746  203.637793    0.000000  120.108554   
SRR8061722  919.319938  663.608498  179.242538    0.264632   82.193479   
SRR8061719   -0.928526    0.011513   -0.048603    0.025274         NaN   

reaction       EX_pi_m    EX_gly_m    EX_uri_m    EX_for_m  EX_lac_L_m  ...  \
SRR8061716  146.042465  140.499660  121.576492  100.583666   93.799670  ...   
SRR8061715   -7.669259    0.405820  -32.735162   -0.243247   -0.457094  ...   
SRR8061718  215.465239    0.031371    3.739775    5.175764   -0.000000  ...   
SRR8061722  150.659936   -0.000000    3.764899   73.883460  107.322748  ...   
SRR8061719   -0.051517   -0.041900   -0.440411   -0.049062   -4.798914  ...   

reaction    EX_sucbz_m  EX_thmmp_m  EX_galactan_m  EX_sbzcoa_m  EX_lichn_m  \
SR

In [ ]:
print(ad_counts)

reaction
EX_4ahmmp_m    11
EX_C02528_m    11
EX_salcn_m     11
EX_asp_L_m     11
EX_glyglu_m    11
               ..
EX_thym_m       1
EX_lyx_L_m      1
EX_btd_RR_m     1
EX_dhpppn_m     1
EX_Lkynr_m      1
Length: 718, dtype: int64


In [ ]:
print(ad_growth)

            community_growth
sample_id                   
SRR8061716         59.082379
SRR8061715         42.397078
SRR8061718         50.645328
SRR8061722         50.510076
SRR8061719         56.031368
SRR8061732         78.579804
SRR8061734         51.908755
SRR8061743         53.121694
SRR8061771         55.626587
SRR8061772         43.427958
SRR8061777         42.786124


In [ ]:
ad_mean_growth = ad_growth["community_growth"].mean()
print(ad_mean_growth)

53.101559012113505


In [ ]:
mci_micom_analysis = run_micom_analysis(mci)
mci_fluxes = mci_micom_analysis[0]
mci_counts = mci_micom_analysis[1]
mci_growth = mci_micom_analysis[2]

1
2
3
4
5
6
7
SRR8061737
SRR8061735
SRR8061764
SRR8061788
SRR8061791
SRR8061799
SRR8061802


In [ ]:
print(mci_fluxes.head())

reaction    EX_prx_mI_m  EX_rpn_104557_m  EX_glypro_m  EX_rsvgluc_m  \
SRR8061737  -407.024913      -406.968150   394.847615    247.576308   
SRR8061735    -0.000000        -0.000000    -0.052191      0.000000   
SRR8061764    -1.572511        -1.471422    -0.272633      1.515790   
SRR8061788    -0.000000        -0.000000    -1.163597     -5.253816   
SRR8061791     0.115920        -0.209790    -0.047131      0.272167   

reaction    EX_3oh_mxn_m  EX_alpz_4oh_glc_m  EX_miso_glc_m  EX_glyasp_m  \
SRR8061737   -185.260851        -170.584699    -170.492160  -163.317876   
SRR8061735     -0.000000           0.000000       0.000000   -56.399766   
SRR8061764     -0.009912          -2.300421       1.687926  -217.969388   
SRR8061788     -0.000000           0.000000       0.000000   -17.145900   
SRR8061791     -0.240289         -22.910741     -97.015493    -0.392141   

reaction    EX_atvacylgluc_m  EX_starch1200_m  ...  EX_dhnacoa_m  \
SRR8061737       -154.989700      -128.974237  ...    

In [ ]:
print(mci_counts)

reaction
EX_1ohmdz_m               7
EX_2ddglcn_m              7
EX_xylan_m                7
EX_xyl_D_m                7
EX_tolcp_glc_m            7
                         ..
EX_turan_D_m              1
EX_mtym_m                 1
EX_dihydro_digitoxin_m    1
EX_dihydro_digoxin_m      1
EX_digitoxin_m            1
Length: 734, dtype: int64


In [ ]:
print(mci_growth)

            community_growth
sample_id                   
SRR8061737         32.318820
SRR8061735         53.032756
SRR8061764         54.016602
SRR8061788         54.623915
SRR8061791         46.175888
SRR8061799         43.077332
SRR8061802         32.831243


In [ ]:
mci_mean_growth = mci_growth["community_growth"].mean()
print(mci_mean_growth)

45.15379372907865


In [ ]:
healthy_micom_analysis = run_micom_analysis(healthy)
healthy_fluxes = healthy_micom_analysis[0]
healthy_counts = healthy_micom_analysis[1]
healthy_growth = healthy_micom_analysis[2]

1
2
3
4
5
6
7
SRR8061749
SRR8061756
SRR8061757
SRR8061753
SRR8061759
SRR8061779
SRR8061782


In [ ]:
print(healthy_fluxes.head())

reaction       EX_ac_m    EX_co2_m  EX_ala_L_m     EX_pi_m   EX_xtsn_m  \
SRR8061749  910.108885  764.504363  227.737413  223.169253  131.285948   
SRR8061756    0.472224   -0.005620  -13.700888   -0.000935    0.042767   
SRR8061757  773.512593  753.557567  200.605765  181.607531  118.333513   
SRR8061753  621.668708  263.765281   10.774254  -25.492031   -0.002585   
SRR8061759   -0.000092   -0.042650   -0.115349 -255.957501    0.012990   

reaction    EX_lac_L_m  EX_glyc3p_m  EX_pro_L_m  EX_man6p_m    EX_o2_m  ...  \
SRR8061749  125.395370   -91.250886  -91.063637  -90.992542 -90.992542  ...   
SRR8061756   -6.792707     0.000000   -1.748568   -0.018031   0.000000  ...   
SRR8061757    8.758546   -76.340558  -75.615878  -75.569044 -77.646846  ...   
SRR8061753   16.287830   -45.771057   -7.155301         NaN -82.657673  ...   
SRR8061759   -0.090543     0.000000    0.101052   -0.066792   0.000000  ...   

reaction    EX_acmp_glc_m  EX_acmp_m  EX_acmpglu_m  EX_adchac_m  \
SRR8061749   

In [ ]:
print(healthy_counts)

reaction
EX_xyl_D_m     7
EX_tchola_m    7
EX_taur_m      7
EX_val_L_m     7
EX_glyasn_m    7
              ..
EX_ocdcea_m    1
EX_ichor_m     1
EX_glutar_m    1
EX_dttp_m      1
EX_n2_m        1
Length: 722, dtype: int64


In [ ]:
print(healthy_growth)

            community_growth
sample_id                   
SRR8061749         55.372845
SRR8061756         34.576294
SRR8061757         50.429086
SRR8061753         46.679581
SRR8061759         36.732941
SRR8061779         55.282700
SRR8061782         49.064293


In [ ]:
healthy_mean_growth = healthy_growth["community_growth"].mean()
print(healthy_mean_growth)

46.876819947289235


Feature Extraction

In [ ]:
# Note: negative flux means secretion, positive means uptake

In [ ]:
# Mean flux
ad_mean = ad_fluxes.mean(axis=0)
mci_mean = mci_fluxes.mean(axis=0)
healthy_mean = healthy_fluxes.mean(axis=0)

In [ ]:
flux_means = pd.DataFrame({
    "AD_mean": ad_mean,
    "MCI_mean": mci_mean,
    "Healthy_mean": healthy_mean})

In [ ]:
# add variance column
flux_means["variance"] = flux_means.max(axis=1) - flux_means.min(axis=1)

In [ ]:
# sort by variance
flux_means = flux_means.sort_values("variance", ascending=False)

In [ ]:
print(flux_means.head(10))

                     AD_mean    MCI_mean  Healthy_mean    variance
reaction                                                          
EX_doh_vcz_glc_m    9.466555  -14.292800    249.570851  263.863651
EX_ac_m           437.400131  186.989298    432.697716  250.410833
EX_co2_m          353.539351  142.936189    353.390275  210.603162
EX_mdzglc_m       -25.576212  -19.748808    181.253280  206.829492
EX_actn_R_m        -1.096614   -5.812334   -195.154097  194.057484
EX_arabinoxyl_m   172.734105    0.326614      0.000031  172.734075
EX_gln_L_m         -0.883617  168.552565     52.169516  169.436182
EX_4ahmmp_m       -16.632100   -0.494175   -166.846751  166.352576
EX_eztmb_glc_m      0.424570    1.679910    166.326851  165.902282
EX_tmao_m          -0.241385   -0.067496    162.947398  163.188783


In [ ]:
# Shorten to top 5
top_flux = flux_means.head(5).index.tolist()
print(top_flux)

['EX_doh_vcz_glc_m', 'EX_ac_m', 'EX_co2_m', 'EX_mdzglc_m', 'EX_actn_R_m']


In [ ]:
ad_top_flux = ad_fluxes[top_flux].copy()
mci_top_flux = mci_fluxes[top_flux].copy()
healthy_top_flux = healthy_fluxes[top_flux].copy()

In [ ]:
# Add community growth column
ad_top_flux["growth"] = ad_growth["community_growth"]
mci_top_flux["growth"] = mci_growth["community_growth"]
healthy_top_flux["growth"] = healthy_growth["community_growth"]

In [ ]:
print("Top 5 Reactions in AD:")
print(ad_top_flux.head())
print("\nTop 5 Reactions in MCI:")
print(mci_top_flux.head())
print("\nTop 5 Reactions in Healthy:")
print(healthy_top_flux.head())

Top 5 Reactions in AD:
reaction    EX_doh_vcz_glc_m     EX_ac_m    EX_co2_m  EX_mdzglc_m  \
SRR8061716               NaN  690.446042  672.789572          NaN   
SRR8061715         -0.299751   -0.146698   99.679088     0.140592   
SRR8061718               NaN  878.207937  667.686746          NaN   
SRR8061722          0.000000  919.319938  663.608498     0.000000   
SRR8061719          0.039858   -0.928526    0.011513    -0.016330   

reaction    EX_actn_R_m     growth  
SRR8061716    -4.454408  59.082379  
SRR8061715          NaN  42.397078  
SRR8061718          NaN  50.645328  
SRR8061722    -0.182889  50.510076  
SRR8061719    -0.117496  56.031368  

Top 5 Reactions in MCI:
reaction    EX_doh_vcz_glc_m     EX_ac_m    EX_co2_m  EX_mdzglc_m  \
SRR8061737        -99.969541    0.030808   -0.013231     0.446429   
SRR8061735          0.000000  699.212360  668.385850     0.000000   
SRR8061764         -0.774721   -0.597348   -0.762384  -140.124472   
SRR8061788          0.000000  700.44828

In [ ]:
# Add condition labels
ad_top_flux["condition"] = "AD"
mci_top_flux["condition"] = "MCI"
healthy_top_flux["condition"] = "Healthy"

In [ ]:
# Combine into one ML dataframe
ml_flux_df = pd.concat([ad_top_flux, mci_top_flux, healthy_top_flux])

In [ ]:
print(ml_flux_df.shape)
ml_flux_df.head()

(25, 7)


reaction,EX_doh_vcz_glc_m,EX_ac_m,EX_co2_m,EX_mdzglc_m,EX_actn_R_m,growth,condition
SRR8061716,NaN,690.446042,672.789572,NaN,-4.454408,59.082379,AD
SRR8061715,-0.299751,-0.146698,99.679088,0.140592,NaN,42.397078,AD
SRR8061718,NaN,878.207937,667.686746,NaN,NaN,50.645328,AD
SRR8061722,0.000000,919.319938,663.608498,0.000000,-0.182889,50.510076,AD
SRR8061719,0.039858,-0.928526,0.011513,-0.016330,-0.117496,56.031368,AD


In [ ]:
# Add sample id column
ml_flux_df["sample_id"] = ml_flux_df.index
cols = ["sample_id", "condition"] + [c for c in ml_flux_df.columns if c not in ["sample_id", "condition"]]
ml_flux_df = ml_flux_df[cols]
ml_flux_df = ml_flux_df.reset_index(drop=True)

In [ ]:
# Create folder for Machine Learning Analysis
ml_analysis_path = "/content/drive/MyDrive/Machine_Learning"
os.makedirs(ml_analysis_path, exist_ok=True)

In [ ]:
# Export MICOM Features to folder
save_path = Path(ml_analysis_path) / "micom_features.csv"
ml_flux_df.to_csv(save_path, index=False)